In [34]:
from utils.torch_siamese_raw import (
    SiameseNetworkLSTM,
    SiameseNetworkConv1D,
    SiameseNetworkConv1DwithBactchNorm,
    SiameseNetworkTransformer,
    SiameseNetworkTransformerUpdated,
    SiameseNetworkMamba,
    SiameseNetworkMambaRes,
    SiameseNetworkMamba2,
    SiameseNetworkMamba3,
    SiameseNetworkMamba3Updated,
    ContrastiveLoss
)

In [52]:
for model_class in [
    SiameseNetworkConv1D,
    SiameseNetworkLSTM,
    SiameseNetworkTransformer,
    SiameseNetworkMamba,]:
    print(model_class.__name__)
    
    model = model_class(input_size=16, embedding_size=32)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Parameters:     {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}\n")

SiameseNetworkConv1D
Total Parameters:     134,624
Trainable Parameters: 134,624

SiameseNetworkLSTM
Total Parameters:     56,352
Trainable Parameters: 56,352

SiameseNetworkTransformer
Total Parameters:     7,104
Trainable Parameters: 7,104

SiameseNetworkMamba
Total Parameters:     450,848
Trainable Parameters: 450,848



In [53]:
import time
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

for model_class in [
    SiameseNetworkConv1D,
    SiameseNetworkLSTM,
    SiameseNetworkTransformer,
    SiameseNetworkMamba,]:
    print(model_class.__name__)

    model = model_class(input_size=16, embedding_size=32).to(device)
    
    model.eval()
    
    batch_size = 32
    dummy_x1 = torch.randn(batch_size, 32, 16).to(device)
    dummy_x2 = torch.randn(batch_size, 32, 16).to(device)
    
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy_x1, dummy_x2)
    
    num_iterations = 1000
    total_time = 0.0
    
    with torch.no_grad():
        for _ in range(num_iterations):
    
            if device.type == "cuda":
                torch.cuda.synchronize()
    
            start_time = time.perf_counter()
            outputs = model(dummy_x1, dummy_x2)
    
            if device.type == "cuda":
                torch.cuda.synchronize() 
    
            end_time = time.perf_counter()
            total_time += end_time - start_time
    
    avg_time_per_batch = (total_time / num_iterations) * 1000
    avg_time_per_sample = avg_time_per_batch / batch_size
    
    print(f"Avg batch processing time ({batch_size} samples): {avg_time_per_batch:.3f} ms")
    print(f"Avg sample processing time:             {avg_time_per_sample:.3f} ms")

cuda
SiameseNetworkConv1D
Avg batch processing time (32 samples): 1.357 ms
Avg sample processing time:             0.042 ms
SiameseNetworkLSTM
Avg batch processing time (32 samples): 1.260 ms
Avg sample processing time:             0.039 ms
SiameseNetworkTransformer
Avg batch processing time (32 samples): 2.092 ms
Avg sample processing time:             0.065 ms
SiameseNetworkMamba
Avg batch processing time (32 samples): 5.698 ms
Avg sample processing time:             0.178 ms


In [33]:
with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    record_shapes=True,
) as prof:
    with torch.no_grad():
        model(dummy_x1, dummy_x2)

# Wydrukuj tabelę posortowaną po czasie wykonania na GPU/CPU
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::linear         0.34%      84.829us        11.38%       2.809ms     351.071us             8  
                                          aten::reshape         0.29%      71.198us         0.53%     130.711us       5.941us            22  
                                             aten::view         0.21%      52.659us         0.21%      52.659us       3.761us            14  
                                                aten::t         0.12%      28.425us         0.26%      63.362us       6.336us            10  
      

In [56]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


def benchmark_training_time(
    model,
    input_size=16,
    sequence_length=32,
    batch_size=32,
    n_epochs=3,
    device="cuda",
):
    device = torch.device(
        device if torch.cuda.is_available() and device == "cuda" else "cpu"
    )
    print(f"Device: {device}")

    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = ContrastiveLoss()

    num_samples = 128
    dummy_x1 = torch.randn(num_samples, sequence_length, input_size)
    dummy_x2 = torch.randn(num_samples, sequence_length, input_size)
    dummy_labels = torch.randint(0, 2, (num_samples,)) * 2

    dataset = TensorDataset(dummy_x1, dummy_x2, dummy_labels)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # warm-up
    model.train()
    for x1, x2, label in train_loader:
        x1, x2, label = x1.to(device), x2.to(device), label.to(device)
        optimizer.zero_grad()
        out1, out2 = model(x1, x2)
        loss = criterion(out1, out2, label)
        loss.backward()
        optimizer.step()

    epoch_times = []
    batch_times = []

    if device.type == "cuda":
        torch.cuda.synchronize()
    start_total = time.perf_counter()

    for epoch in range(n_epochs):
        if device.type == "cuda":
            torch.cuda.synchronize()
        start_epoch = time.perf_counter()

        train_loss = 0
        for x1, x2, label in train_loader:
            if device.type == "cuda":
                torch.cuda.synchronize()
            start_batch = time.perf_counter()

            x1, x2, label = x1.to(device), x2.to(device), label.to(device)
            optimizer.zero_grad()
            out1, out2 = model(x1, x2)
            loss = criterion(out1, out2, label)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

            if device.type == "cuda":
                torch.cuda.synchronize()
            end_batch = time.perf_counter()

            batch_times.append(end_batch - start_batch)

        epoch_loss = train_loss / len(train_loader)

        if device.type == "cuda":
            torch.cuda.synchronize()
        end_epoch = time.perf_counter()

        epoch_duration = end_epoch - start_epoch
        epoch_times.append(epoch_duration)

    if device.type == "cuda":
        torch.cuda.synchronize()
    end_total = time.perf_counter()

    total_time = end_total - start_total
    avg_epoch_time = sum(epoch_times) / len(epoch_times)

    avg_batch_time_ms = (sum(batch_times) / len(batch_times)) * 1000

    print("=" * 40)
    print(f"Total time for ({n_epochs} epochs): {total_time:.4f} s")
    print(f"Average epoch time:   {avg_epoch_time:.4f} s")
    print(f"Average batch time:   {avg_batch_time_ms:.3f} ms")  # <--- WYDRUK
    print("=" * 40 + "\n")

    return total_time, avg_epoch_time, avg_batch_time_ms


for model_class in [
    SiameseNetworkConv1D,
    SiameseNetworkLSTM,
    SiameseNetworkTransformer,
    SiameseNetworkMamba,]:
    print(model_class.__name__)
    
    model = model_class(input_size=16, embedding_size=32)

    benchmark_training_time(
        model=model,
        input_size=16,
        sequence_length=32,
        batch_size=32,
        n_epochs=100,
        device="cuda",
    )

SiameseNetworkConv1D
Device: cuda
Total time for (100 epochs): 2.3334 s
Average epoch time:   0.0233 s
Average batch time:   5.365 ms

SiameseNetworkLSTM
Device: cuda
Total time for (100 epochs): 2.3230 s
Average epoch time:   0.0232 s
Average batch time:   5.364 ms

SiameseNetworkTransformer
Device: cuda
Total time for (100 epochs): 5.1884 s
Average epoch time:   0.0519 s
Average batch time:   12.493 ms

SiameseNetworkMamba
Device: cuda
Total time for (100 epochs): 7.3317 s
Average epoch time:   0.0733 s
Average batch time:   17.766 ms

